# 4. Modeling and Tuning

- Load pre-split train and test sets
- Train baseline models: Logistic Regression, Decision Tree, Random Forest, XGBoost, Extreme Boosting
- Tune each model with cross-validation
- Save comparison results

In [15]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold

ROOT = Path(globals()["__vsc_ipynb_file__"]).parent.parent
TRAIN_PATH = ROOT / "data" / "processed" / "train.csv"
TEST_PATH = ROOT / "data" / "processed" / "test.csv"
RESULTS_PATH = ROOT / "results" / "model_comparison.csv"
BEST_MODEL_CONFIG_PATH = ROOT / "results" / "best_model_config.json"

# Load pre-split data (already encoded and feature-selected)
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

y_train = train_df["TARGET_CLASS"].astype(int)
y_test = test_df["TARGET_CLASS"].astype(int)
X_train = train_df.drop(columns=["TARGET_CLASS"]).copy()
X_test = test_df.drop(columns=["TARGET_CLASS"]).copy()

# Encode target to 0-indexed for XGBoost compatibility
target_le = LabelEncoder()
y_train = pd.Series(target_le.fit_transform(y_train), index=y_train.index)
y_test = pd.Series(target_le.transform(y_test), index=y_test.index)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target classes: {target_le.classes_} -> {list(range(len(target_le.classes_)))}")

Train: (1464, 14), Test: (366, 14)
Target classes: [1 2 3 4] -> [0, 1, 2, 3]


In [16]:
# Baseline models
models = {
    "LogisticRegression": LogisticRegression(max_iter=5000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(
        objective="multi:softprob", eval_metric="mlogloss",
        n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42,
    ),
    "ExtremeBoosting": XGBClassifier(
        objective="multi:softprob", eval_metric="mlogloss",
        n_estimators=500, learning_rate=0.03, max_depth=8,
        min_child_weight=3, reg_lambda=1.5, random_state=42,
    ),
}

def score_model(name, variant, model):
    y_pred = model.predict(X_test)
    return {
        "model": name, "variant": variant,
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision_macro": float(precision_score(y_test, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
    }

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    rows.append(score_model(name, "baseline", model))
    print(f"  {name} baseline: f1_macro={rows[-1]['f1_macro']:.4f}")

results_df = pd.DataFrame(rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)
results_df

  LogisticRegression baseline: f1_macro=0.3053
  DecisionTree baseline: f1_macro=0.3275
  RandomForest baseline: f1_macro=0.3365
  XGBoost baseline: f1_macro=0.3222
  ExtremeBoosting baseline: f1_macro=0.3360


,model,variant,accuracy,precision_macro,recall_macro,f1_macro
0,RandomForest,baseline,0.445355,0.386509,0.331090,0.336502
1,ExtremeBoosting,baseline,0.420765,0.363962,0.328576,0.335965
2,DecisionTree,baseline,0.377049,0.324503,0.337062,0.327507
3,XGBoost,baseline,0.407104,0.343066,0.316565,0.322242
4,LogisticRegression,baseline,0.442623,0.347166,0.312934,0.305313


In [17]:
# Hyperparameter tuning
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
best_params = {}

# Decision Tree
dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {"max_depth": [5, 8, None], "min_samples_split": [2, 10], "min_samples_leaf": [1, 4], "criterion": ["gini", "entropy"]},
    scoring="f1_macro", cv=cv, n_jobs=-1, refit=True,
)
dt_grid.fit(X_train, y_train)
rows.append(score_model("DecisionTree", "tuned", dt_grid.best_estimator_))
best_params["DecisionTree"] = dt_grid.best_params_

# Logistic Regression
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=7000, random_state=42),
    {"C": [0.1, 1.0, 2.0], "solver": ["lbfgs", "newton-cg"], "class_weight": [None, "balanced"]},
    scoring="f1_macro", cv=cv, n_jobs=-1, refit=True,
)
lr_grid.fit(X_train, y_train)
rows.append(score_model("LogisticRegression", "tuned", lr_grid.best_estimator_))
best_params["LogisticRegression"] = lr_grid.best_params_

# Random Forest
rf_grid = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {"n_estimators": [300, 500, 700], "max_depth": [None, 10, 16], "min_samples_split": [2, 5, 10],
     "min_samples_leaf": [1, 2, 4], "max_features": ["sqrt", "log2", 0.5], "class_weight": [None, "balanced"]},
    n_iter=16, scoring="f1_macro", cv=cv, n_jobs=-1, refit=True, random_state=42,
)
rf_grid.fit(X_train, y_train)
rows.append(score_model("RandomForest", "tuned", rf_grid.best_estimator_))
best_params["RandomForest"] = rf_grid.best_params_

# XGBoost
xgb_grid = RandomizedSearchCV(
    XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", n_estimators=250, random_state=42),
    {"max_depth": [3, 5, 7], "learning_rate": [0.03, 0.05, 0.1], "subsample": [0.8, 0.9, 1.0],
     "colsample_bytree": [0.8, 0.9, 1.0], "min_child_weight": [1, 3, 5]},
    n_iter=12, scoring="f1_macro", cv=cv, n_jobs=-1, refit=True, random_state=42,
)
xgb_grid.fit(X_train, y_train)
rows.append(score_model("XGBoost", "tuned", xgb_grid.best_estimator_))
best_params["XGBoost"] = xgb_grid.best_params_

# Extreme Boosting
xtreme_grid = RandomizedSearchCV(
    XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", n_estimators=200, random_state=42),
    {"max_depth": [5, 7, 9], "learning_rate": [0.01, 0.03, 0.05], "subsample": [0.8, 0.9, 1.0],
     "colsample_bytree": [0.7, 0.8, 0.9], "min_child_weight": [1, 3, 5],
     "gamma": [0.0, 0.1, 0.3], "reg_lambda": [1.0, 1.5, 2.0]},
    n_iter=18, scoring="f1_macro", cv=cv, n_jobs=-1, refit=True, random_state=42,
)
xtreme_grid.fit(X_train, y_train)
rows.append(score_model("ExtremeBoosting", "tuned", xtreme_grid.best_estimator_))
best_params["ExtremeBoosting"] = xtreme_grid.best_params_

# Combine and save
results_df = pd.DataFrame(rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)
best_row = results_df.iloc[0]

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(RESULTS_PATH, index=False)

best_config = {"model": str(best_row["model"]), "variant": str(best_row["variant"]),
               "params": best_params.get(str(best_row["model"]), {})}
BEST_MODEL_CONFIG_PATH.write_text(json.dumps(best_config, indent=2, default=str), encoding="utf-8")

print(f"\nBest model: {best_row['model']} ({best_row['variant']}) — f1_macro={best_row['f1_macro']:.4f}")
print(f"Saved: {RESULTS_PATH}")
results_df


Best model: RandomForest (tuned) — f1_macro=0.3553
Saved: c:\Users\Johannes Espeset\Downloads\Master-Class-main\Master-Class-main\results\model_comparison.csv


,model,variant,accuracy,precision_macro,recall_macro,f1_macro
0,RandomForest,tuned,0.385246,0.349060,0.367603,0.355294
1,LogisticRegression,tuned,0.368852,0.367085,0.429841,0.354013
2,ExtremeBoosting,tuned,0.437158,0.393022,0.335548,0.345053
3,RandomForest,baseline,0.445355,0.386509,0.331090,0.336502
4,ExtremeBoosting,baseline,0.420765,0.363962,0.328576,0.335965
5,XGBoost,tuned,0.426230,0.360475,0.325375,0.331475
6,DecisionTree,baseline,0.377049,0.324503,0.337062,0.327507
7,DecisionTree,tuned,0.382514,0.321618,0.325435,0.322812
8,XGBoost,baseline,0.407104,0.343066,0.316565,0.322242
9,LogisticRegression,baseline,0.442623,0.347166,0.312934,0.305313
